# Tiny Agent with Tools ?

All open source: tiny local model, wiki library, no API keys. Run top-to-bottom.

In [ ]:
!pip install -q smolagents[transformers] wikipedia

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.4/148.4 kB 9.1 MB/s eta 0:00:00


## 1) Define KB

In [ ]:
#To-Do: you can add your own knowledge base snippets here
kb_snippets = [
    {'source': 'kb:agentic', 'text': 'Agentic AI loops plan, choose tools, and reflect before answering.'},
    {'source': 'kb:tools', 'text': 'Useful tools: math, search, and domain-specific lookup.'},
    {'source': 'kb:citation', 'text': 'Always cite where evidence came from to stay transparent.'},
    {'source': 'kb:brevity', 'text': 'Keep answers concise (2-4 sentences).'},
    {'source': 'kb:followup', 'text': 'If evidence is missing, say so and propose a follow-up question.'},
]
print('KB entries:', len(kb_snippets))


KB entries: 5


## 2) Define tools

In [ ]:
from smolagents import Tool, TransformersModel, ToolCallingAgent

class KBLookupTool(Tool):
    name = "kb_lookup_tool"
    description = "Looks up relevant information from a custom knowledge base."
    inputs = {"query": {"type": "string", "description": "search query"}}
    output_type = "string"

    def __init__(self, kb):
        super().__init__()
        self.kb = kb

    def forward(self, query: str) -> str:
        q = query.lower()
        matches = [
            f"[{item['source']}] {item['text']}"
            for item in self.kb
            if any(w in item["text"].lower() for w in q.split())
        ]
        return "".join(matches) if matches else "No KB match."


class MathTool(Tool):
    name = "math_tool"
    description = "Add or multiply two numbers."
    inputs = {
        "a": {"type": "number", "description": "first number"},
        "b": {"type": "number", "description": "second number"},
        "op": {"type": "string", "description": "operation: add or multiply"}
    }
    output_type = "string"

    def forward(self, a: float, b: float, op: str = "add") -> str:
        if op == "multiply":
            return str(a * b)
        return str(a + b)


kb_tool = KBLookupTool(kb_snippets)
math_tool = MathTool()

## 3) Model (tiny local)

In [ ]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

model = TransformersModel(
    model_id=MODEL_ID
)

print("Model ready:", MODEL_ID)

## 4) Agent

In [ ]:
agent = ToolCallingAgent(
    tools=[kb_tool, math_tool],
    model=model,
    max_steps=2,
    instructions="You are an agentic AI that uses tools to answer questions.",
)

print(agent)

## 5) Test queries

In [ ]:
tests = [
    "Add 12 and 30.",
    "Multiply 7 by 6.",
    "What is an agentic AI loop?",
]

for q in tests:
    print("---")
    print("Q:", q)
    result = agent.run(q)
    print("Answer:", result)